<a href="https://colab.research.google.com/github/ahmad1bakundi-ops/data-engineering-journey/blob/main/lesson_4_filtering_and_pivots.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
DATA_PATH = '/content/drive/MyDrive/data-engineering-journey/lesson-02-data/'
df = pd.read_csv(DATA_PATH + 'nigerian_ecommerce_sales_CLEAN.csv')
df['order_date'] = pd.to_datetime(df['order_date'])
df['revenue'] = df['quantity'] * df['unit_price_ngn']
df.shape

(724, 12)

In [3]:
df.columns

Index(['order_id', 'customer_id', 'customer_name', 'customer_state', 'product',
       'category', 'quantity', 'unit_price_ngn', 'order_date',
       'payment_method', 'status', 'revenue'],
      dtype='object')

In [4]:
delivered = df[df['status'] == 'Delivered']
delivered.shape

(150, 12)

In [5]:
delivered['revenue'].sum()

np.float64(11649829.98)

In [6]:
df['revenue'].sum()

np.float64(72386244.09)

In [7]:
df.groupby('status').agg(
    order_count=('order_id', 'count'),
    total_revenue=('revenue', 'sum'),
    avg_order_value=('revenue', 'mean'),
).sort_values('total_revenue', ascending=False)

,order_count,total_revenue,avg_order_value
status,,,
Returned,143,19484624.20,136256.113287
Cancelled,159,16353148.84,102849.992704
Shipped,126,12478071.88,99032.316508
Pending,146,12420569.19,85072.391712
Delivered,150,11649829.98,77665.533200


In [8]:
df[df['status'] == 'Delivered'].groupby('category')['revenue'].sum().sort_values(ascending=False)

,revenue
category,
Electronics,6812597.17
Home Appliances,1569072.29
Furniture,1474531.80
Fashion,980379.52
Groceries,446626.66
Baby Care,245445.22
Books,121177.32


In [9]:
q4_delivered = df[
    (df['status'] == 'Delivered') &
    (df['order_date'] >= '2024-10-01')
]
q4_delivered['revenue'].sum()

np.float64(4054476.97)

In [11]:
df.groupby(['customer_state', 'category'])['revenue'].sum().head(20)

customer_state  category       
Aba             Baby Care            42180.24
                Books                 8177.18
                Electronics        7519476.05
                Fashion             350577.11
                Furniture          1366085.34
                Groceries           175436.60
                Home Appliances     540330.23
Abuja           Baby Care           100087.82
                Books                 3715.70
                Electronics        1882058.82
                Fashion             395830.24
                Furniture           523189.43
                Groceries           145820.73
                Home Appliances     510797.87
Benin City      Baby Care             8897.84
                Books                56606.58
                Electronics        2654958.30
                Fashion             106665.38
                Furniture            82545.52
                Groceries            80827.55
Name: revenue, dtype: float64

In [12]:
df.pivot_table(
    index='customer_state',
    columns='category',
    values='revenue',
    aggfunc='sum',
    fill_value=0
)

category,Baby Care,Books,Electronics,Fashion,Furniture,Groceries,Home Appliances
customer_state,,,,,,,
Aba,42180.24,8177.18,7519476.05,350577.11,1366085.34,175436.60,540330.23
Abuja,100087.82,3715.70,1882058.82,395830.24,523189.43,145820.73,510797.87
Benin City,8897.84,56606.58,2654958.30,106665.38,82545.52,80827.55,1534489.40
Calabar,37617.24,31156.39,1405952.11,375395.99,227639.95,88515.46,288577.99
Enugu,68022.53,81900.28,4520280.60,520216.43,894810.58,228346.95,733252.41
Ibadan,97597.77,72462.43,2674748.16,306365.23,463896.14,115328.43,337257.27
Ilorin,71897.54,13905.89,889542.21,394562.04,142985.58,64360.10,769533.37
Jos,159616.43,39173.96,2285767.75,702619.04,496448.04,68165.05,966615.33
Kaduna,91593.11,106032.08,4377095.18,445816.04,612727.94,171272.18,728216.16


In [13]:
df.pivot_table(
    index='customer_state',
    columns='category',
    values='revenue',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
)

category,Baby Care,Books,Electronics,Fashion,Furniture,Groceries,Home Appliances,Total
customer_state,,,,,,,,
Aba,42180.24,8177.18,7519476.05,350577.11,1366085.34,175436.60,540330.23,10002262.75
Abuja,100087.82,3715.70,1882058.82,395830.24,523189.43,145820.73,510797.87,3561500.61
Benin City,8897.84,56606.58,2654958.30,106665.38,82545.52,80827.55,1534489.40,4524990.57
Calabar,37617.24,31156.39,1405952.11,375395.99,227639.95,88515.46,288577.99,2454855.13
Enugu,68022.53,81900.28,4520280.60,520216.43,894810.58,228346.95,733252.41,7046829.78
Ibadan,97597.77,72462.43,2674748.16,306365.23,463896.14,115328.43,337257.27,4067655.43
Ilorin,71897.54,13905.89,889542.21,394562.04,142985.58,64360.10,769533.37,2346786.73
Jos,159616.43,39173.96,2285767.75,702619.04,496448.04,68165.05,966615.33,4718405.60
Kaduna,91593.11,106032.08,4377095.18,445816.04,612727.94,171272.18,728216.16,6532752.69


In [14]:
state_category = df.pivot_table(
    index='customer_state',
    columns='category',
    values='revenue',
    aggfunc='sum',
    fill_value=0
)

# Convert each row to percentages of that row's total
state_category_pct = state_category.div(state_category.sum(axis=1), axis=0) * 100
state_category_pct.round(1)

category,Baby Care,Books,Electronics,Fashion,Furniture,Groceries,Home Appliances
customer_state,,,,,,,
Aba,0.4,0.1,75.2,3.5,13.7,1.8,5.4
Abuja,2.8,0.1,52.8,11.1,14.7,4.1,14.3
Benin City,0.2,1.3,58.7,2.4,1.8,1.8,33.9
Calabar,1.5,1.3,57.3,15.3,9.3,3.6,11.8
Enugu,1.0,1.2,64.1,7.4,12.7,3.2,10.4
Ibadan,2.4,1.8,65.8,7.5,11.4,2.8,8.3
Ilorin,3.1,0.6,37.9,16.8,6.1,2.7,32.8
Jos,3.4,0.8,48.4,14.9,10.5,1.4,20.5
Kaduna,1.4,1.6,67.0,6.8,9.4,2.6,11.1


In [15]:
state_category.sort_values('Electronics', ascending=False)

category,Baby Care,Books,Electronics,Fashion,Furniture,Groceries,Home Appliances
customer_state,,,,,,,
Aba,42180.24,8177.18,7519476.05,350577.11,1366085.34,175436.60,540330.23
Port Harcourt,148255.09,76227.86,5096645.25,74063.41,130011.47,160679.34,318110.16
Enugu,68022.53,81900.28,4520280.60,520216.43,894810.58,228346.95,733252.41
Kaduna,91593.11,106032.08,4377095.18,445816.04,612727.94,171272.18,728216.16
Onitsha,131338.90,45708.03,4041159.88,243495.37,523431.31,57020.52,1373527.97
Ibadan,97597.77,72462.43,2674748.16,306365.23,463896.14,115328.43,337257.27
Benin City,8897.84,56606.58,2654958.30,106665.38,82545.52,80827.55,1534489.40
Jos,159616.43,39173.96,2285767.75,702619.04,496448.04,68165.05,966615.33
Abuja,100087.82,3715.70,1882058.82,395830.24,523189.43,145820.73,510797.87
